# Finance Sentiment Intelligence - End-to-End Pipeline

This notebook ties together all the modular scripts from the `src/` directory into a single interactive pipeline.

In [1]:
# ================================
# 1. Imports
# ================================
import os
import sys
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")


# ================================
# 2. Fix Python Path (ROBUST)
# ================================
def setup_project_path():
    """
    Dynamically locate project root (folder containing 'src')
    and add it to sys.path for clean imports.
    Works regardless of where notebook is executed from.
    """
    current = os.getcwd()

    while True:
        # Check if 'src' folder exists here
        if os.path.isdir(os.path.join(current, "src")):
            if current not in sys.path:
                sys.path.insert(0, current)  # insert at front (priority)
            print(f"✅ Project root set to: {current}")
            return current

        # Move one level up
        parent = os.path.dirname(current)

        # If reached filesystem root → stop
        if parent == current:
            raise RuntimeError("❌ Could not find project root (no 'src' folder found)")

        current = parent


PROJECT_ROOT = setup_project_path()


# ================================
# 3. Safe Imports (after path fix)
# ================================
try:
    from src.preprocessing.cleaner import full_clean
    from src.preprocessing.wrangler import (
        drop_nulls_and_duplicates,
        filter_short_texts,
        encode_sentiment
    )

    from src.features.feature_engineer import build_tfidf_matrix

    from src.models.train import train_all_models
    from src.models.evaluator import (
        print_model_report,
        build_comparison_table
    )

    from src.visualization.visualizer import (
        plot_sentiment_distribution,
        plot_wordclouds
    )

    print("✅ All imports successful!")

except Exception as e:
    print("❌ Import failed:")
    print(e)
    raise


# ================================
# 4. Optional Debug Info
# ================================
print("\n📁 Current Working Directory:", os.getcwd())
print("📦 Python Path (first 3):", sys.path[:3])

✅ Project root set to: c:\A-Projects gng\sentiment analysis\finance-sentiment-intelligence
❌ Import failed:
No module named 'nltk'


ModuleNotFoundError: No module named 'nltk'

## 1. Load Data & Preprocessing
We begin by reading the raw CSV. We then clean the text (removing URLs, emojis, HTML tags) and filter out duplicate or overly short posts.

In [8]:
data_path = os.path.join(PROJECT_ROOT, "data", "raw", "reddit_posts.csv")
if not os.path.exists(data_path):
    print(f"Could not find {data_path}. Ensure the scraper has run first!")
else:
    df = pd.read_csv(data_path)
    print("Raw Data Shape:", df.shape)
    display(df[['subreddit', 'text', 'sentiment']].head(3))

    print("\n--- Cleaning Data ---")
    # 1. Clean the raw text column
    df['clean_text'] = df['text'].apply(lambda x: full_clean(str(x)))
    
    # 2. Data Wrangling
    df = drop_nulls_and_duplicates(df)
    df = filter_short_texts(df, min_words=3)
    
    # 3. Label Encoding
    # encode_sentiment expects the target column to output 'sentiment_encoded' (0,1,2)
    df = encode_sentiment(df)
    
    print("Clean Data Shape:", df.shape)
    display(df[['subreddit', 'clean_text', 'sentiment_encoded']].head(3))

Raw Data Shape: (1140, 10)


,subreddit,text,sentiment
0,stocks,r/Stocks Daily Discussion &amp; Fundamentals F...,positive
1,stocks,"Gold Plunges 1.6%! The $3,300 Make-or-Break Ba...",negative
2,stocks,Are we following Argentina's footsteps? Trump ...,negative



--- Cleaning Data ---
2026-04-20 11:31:51 | INFO | src.preprocessing.wrangler | Dropped nulls/duplicates: 62 rows removed


KeyError: 'cleaned_text'

## 2. Feature Engineering
Convert the cleaned text into a sparse TF-IDF numeric matrix that algorithms can understand.

In [ ]:
print("Building TF-IDF Matrix...")
X_tfidf, vectorizer = build_tfidf_matrix(df, text_column='clean_text', max_features=5000)
print("TF-IDF Shape:", X_tfidf.shape)

y = df['sentiment_encoded'] if 'sentiment_encoded' in df.columns else df['sentiment']
print("\nTarget Distribution:")
print(y.value_counts())

## 3. Model Training
Train Logistic Regression, Naive Bayes, and Random Forest on the dataset.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print("Training Models. This might take a few moments...")
results = train_all_models(X_train, y_train, X_test, y_test)

for model_name, data in results.items():
    print(f"\n{'='*40}\nEvaluating {model_name}\n{'='*40}")
    print_model_report(model_name, data['y_test'], data['y_pred'])

## 4. Evaluation & Visualizations
Compare the model performance and plot engaging text analysis visuals natively.

In [ ]:
comparison_df = build_comparison_table(results)
display(comparison_df)

# Native inline plot for accuracies
plt.figure(figsize=(10,6))
sns.barplot(data=comparison_df.reset_index(), x='index', y='Accuracy', palette='viridis')
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.ylim(0, 1.0)
plt.show()

print("\nGenerating Sentiment Distribution Plot...")
plot_sentiment_distribution(df)

print("\nGenerating Wordclouds...")
plot_wordclouds(df)